## Exploración y Análisis de ERA5 · Extracción (Chile · 2000-2025)

Extracción desde el **ARCO ERA5** de Google Cloud con `xarray` + `Dask`

## 1. Configuración inicial

In [1]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from dask.distributed import Client, progress

### Cluster de Dask

El trabajo es E/S de red, así que conviene un solo worker en proceso con muchos hilos
en vez de varios procesos.

In [2]:
from dask.distributed import get_client

# Cierra cualquier cluster Dask previo (p. ej. si esta celda se re-ejecuta sin
# haber pasado por la sección 9), para no acumular clusters huérfanos.
try:
    get_client().close()
    print('Cluster previo detectado y cerrado.')
except ValueError:
    pass


In [3]:
client = Client(processes=False, n_workers=1, threads_per_worker=32)
client

## 2. Apertura del store

In [4]:
URL_ARCO = 'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3'

ds = xr.open_zarr(
    URL_ARCO,
    chunks=None,
    storage_options={'token': 'anon'},
)

print(f'Store abierto: {URL_ARCO}')
variables = list(ds.data_vars)
print(f'Variables disponibles ({len(variables)}):')
for varname_store in variables:
    print(varname_store)
print(f'Dimensiones: {dict(ds.sizes)}')

# El eje time del store se extiende más allá de los datos válidos (NaN a futuro).
print(f"valid_time_start: {ds.attrs.get('valid_time_start')}")
print(f"valid_time_stop:  {ds.attrs.get('valid_time_stop')}")

Store abierto: gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3
Variables disponibles (273):
100m_u_component_of_wind
100m_v_component_of_wind
10m_u_component_of_neutral_wind
10m_u_component_of_wind
10m_v_component_of_neutral_wind
10m_v_component_of_wind
10m_wind_gust_since_previous_post_processing
2m_dewpoint_temperature
2m_temperature
air_density_over_the_oceans
angle_of_sub_gridscale_orography
anisotropy_of_sub_gridscale_orography
benjamin_feir_index
boundary_layer_dissipation
boundary_layer_height
charnock
clear_sky_direct_solar_radiation_at_surface
cloud_base_height
coefficient_of_drag_with_waves
convective_available_potential_energy
convective_inhibition
convective_precipitation
convective_rain_rate
convective_snowfall
convective_snowfall_rate_water_equivalent
downward_uv_radiation_at_the_surface
duct_base_height
eastward_gravity_wave_surface_stress
eastward_turbulent_surface_stress
evaporation
forecast_albedo
forecast_logarithm_of_surface_roughness_for_heat
f

## 3. Parámetros

La carpeta base puede sobrescribirse con la variable de entorno `ERA5_DIR` (la misma que usa `era5_agregacion_precipitacion.ipynb`).

In [5]:
# Variables a extraer desde el store (base). Las derivadas (viento, HR) se calculan en la sección 7.
#   total_precipitation  -> precipitación total          2m_temperature -> temperatura a 2 m
#   10m_u_component_of_wind -> viento zonal a 10 m          10m_v_component_of_wind -> viento meridional a 10 m
#   2m_dewpoint_temperature -> punto de rocío a 2 m (para humedad relativa)
VARIABLES = ['total_precipitation', '2m_temperature', '10m_u_component_of_wind', '10m_v_component_of_wind', '2m_dewpoint_temperature']

# Región (Todo Chile)
LAT_NORTE, LAT_SUR = -17.0, -56.0
LON_OESTE, LON_ESTE = -77.0, -66.0

# Período temporal
ANIO_INI, ANIO_FIN = 2000, 2025

# El store declara hasta cuándo hay datos válidos; más allá el eje time es NaN.
# tp necesita además las 00:00 del 1-ene siguiente.
_valid_stop = ds.attrs.get('valid_time_stop')
if _valid_stop is not None:
    _ultimo_necesario = pd.Timestamp(f'{ANIO_FIN + 1}-01-01 00:00')
    if _ultimo_necesario > pd.Timestamp(_valid_stop):
        raise ValueError(f'ANIO_FIN={ANIO_FIN} supera los datos válidos del store '
                         f'(valid_time_stop={_valid_stop})')
else:
    print('Aviso: el store no declara valid_time_stop; no se pudo validar ANIO_FIN.')

# Submuestreo (1 = serie horaria completa; 3 o 6 reduce descarga)
PASO_HORAS = 1
HORAS_POR_TAREA = 24
PASOS_POR_TAREA = max(1, HORAS_POR_TAREA // PASO_HORAS)

# Directorios (ruta absoluta: no depende del directorio de trabajo del kernel;
# se puede cambiar con la variable de entorno ERA5_DIR)
DIR_SALIDA = Path(os.environ.get('ERA5_DIR', r'D:\Documents\Proyectos\era5'))
DIR_CACHE = DIR_SALIDA / "cache_era5_chile"
DIR_CACHE.mkdir(parents=True, exist_ok=True)


In [6]:
# Revisión de metadatos de las variables solicitadas en el store ERA5
for var in VARIABLES:
    if var not in ds.data_vars:
        raise ValueError(f'Variable {var} no encontrada en el store ERA5.')
    first_view = ds[var].isel(time=0)
    print(f'Metadatos de la variable {var}:')
    print(first_view.attrs)


Metadatos de la variable total_precipitation:
{'long_name': 'Total precipitation', 'short_name': 'tp', 'units': 'm'}
Metadatos de la variable 2m_temperature:
{'long_name': '2 metre temperature', 'short_name': 't2m', 'units': 'K'}
Metadatos de la variable 10m_u_component_of_wind:
{'long_name': '10 metre U wind component', 'short_name': 'u10', 'units': 'm s**-1'}
Metadatos de la variable 10m_v_component_of_wind:
{'long_name': '10 metre V wind component', 'short_name': 'v10', 'units': 'm s**-1'}
Metadatos de la variable 2m_dewpoint_temperature:
{'long_name': '2 metre dewpoint temperature', 'short_name': 'd2m', 'units': 'K'}


## 4. Definiciones de variables

Cada variable tiene metadata específica: nombre en el store, conversión de unidades y
prefijo de caché. La agregación temporal se decide en la sección 7.

In [8]:
VAR_CONFIG = {
    'total_precipitation': {
        'store_name': 'total_precipitation',
        'short_name': 'tp',
        'long_name': 'Total precipitation',
        'units': 'mm',
        'convert': lambda x: (x.clip(min=0) * 1000),  # m -> mm, recorta ruido negativo
        'cache_prefix': 'tp_',
    },
    '2m_temperature': {
        'store_name': '2m_temperature',
        'short_name': 't2m',
        'long_name': '2 metre temperature',
        'units': '°C',
        'convert': lambda x: (x - 273.15),  # K -> °C
        'cache_prefix': 't2m_',
    },
    '10m_u_component_of_wind': {
        'store_name': '10m_u_component_of_wind',
        'short_name': 'u10',
        'long_name': '10 metre U wind component',
        'units': 'm/s',
        'convert': lambda x: x,  # ya viene en m/s
        'cache_prefix': 'u10_',
    },
    '10m_v_component_of_wind': {
        'store_name': '10m_v_component_of_wind',
        'short_name': 'v10',
        'long_name': '10 metre V wind component',
        'units': 'm/s',
        'convert': lambda x: x,  # ya viene en m/s
        'cache_prefix': 'v10_',
    },
    '2m_dewpoint_temperature': {
        'store_name': '2m_dewpoint_temperature',
        'short_name': 'd2m',
        'long_name': '2 metre dewpoint temperature',
        'units': '°C',
        'convert': lambda x: (x - 273.15),  # K -> °C
        'cache_prefix': 'd2m_',
    },
}

# VAR_CONFIG usa el nombre del store; la parte diaria trabaja con nombres cortos (tp, t2m, ...)
VAR_CFG = {cfg['short_name']: cfg for cfg in VAR_CONFIG.values()}

print(f'Variables a extraer: {VARIABLES}')
print(f'Período: {ANIO_INI}-{ANIO_FIN}')
print(f'Región: lat {LAT_NORTE}..{LAT_SUR}, lon {LON_OESTE}..{LON_ESTE}')

Variables a extraer: ['total_precipitation', '2m_temperature', '10m_u_component_of_wind', '10m_v_component_of_wind', '2m_dewpoint_temperature']
Período: 2000-2025
Región: lat -17.0..-56.0, lon -77.0..-66.0


## 5. Funciones de recorte y procesamiento

In [9]:
def recorte_horario(anio: int, variable: str):
    '''Recorte lazy de un año, con unidades convertidas. No descarga nada.'''
    cfg = VAR_CONFIG[variable]
    
    # Manejo dinámico de timestamps según el comportamiento de la variable
    if cfg['short_name'] == 'tp':
        time_start = f'{anio}-01-01 00:00'
        time_end = f'{anio + 1}-01-01 00:00'
    else:
        time_start = f'{anio}-01-01 00:00'
        time_end = f'{anio}-12-31 23:00'

    da = ds[cfg['store_name']].sel(
        latitude=slice(LAT_NORTE, LAT_SUR),
        longitude=slice(LON_OESTE % 360, LON_ESTE % 360),
        time=slice(time_start, time_end),
    )
    
    if PASO_HORAS > 1:
        da = da.isel(time=slice(None, None, PASO_HORAS))

    da = da.chunk({'time': PASOS_POR_TAREA, 'latitude': -1, 'longitude': -1})
    da = cfg['convert'](da).rename(cfg['short_name'])
    da.attrs = {
        'long_name': cfg['long_name'],
        'short_name': cfg['short_name'],
        'units': cfg['units'],
    }
    return da

def computar_con_reintentos(obj, intentos=4, espera=60):
    '''Computa un objeto perezoso reintentando ante cortes de red.'''
    for intento in range(1, intentos + 1):
        try:
            fut = obj.persist(retries=3)
            progress(fut)
            return fut.compute(retries=3)
        except Exception as exc:
            print(f'  intento {intento}/{intentos} falló: {exc!r}')
            if intento == intentos:
                raise
            print(f'  reintentando en {espera * intento} s...')
            time.sleep(espera * intento)


print('Funciones definidas.')

Funciones definidas.


## 6. Descarga con caché incremental por año

In [10]:
# Restos de una ejecución abortada
for f in DIR_CACHE.glob('*.tmp.nc'):
    f.unlink()

for variable in VARIABLES:
    cfg = VAR_CONFIG[variable]
    print(f'\n=== Extrayendo {cfg["long_name"]} ===')

    # el encoding depende solo de la variable, no del año: se calcula una vez
    enc = {cfg['short_name']: {'zlib': True, 'complevel': 4, 'dtype': 'float32'}}

    for anio in range(ANIO_INI, ANIO_FIN + 1):
        destino = DIR_CACHE / f'{cfg["cache_prefix"]}{anio}.nc'
        if destino.exists():
            print(f'{anio}: ya en caché ({destino.stat().st_size / 1e6:.1f} MB)')
            continue

        print(f'{anio}: descargando...')
        t0 = time.time()
        datos = computar_con_reintentos(recorte_horario(anio, variable))

        tmp = destino.with_suffix('.tmp.nc')
        datos.to_netcdf(tmp, engine='netcdf4', encoding=enc)
        tmp.replace(destino)
        print(f'{anio}: listo en {time.time() - t0:.0f} s ({datos.sizes["time"]} pasos)')


=== Extrayendo Total precipitation ===
2000: ya en caché (98.7 MB)
2001: ya en caché (96.3 MB)
2002: ya en caché (97.5 MB)
2003: ya en caché (92.8 MB)
2004: ya en caché (95.0 MB)
2005: ya en caché (95.0 MB)
2006: ya en caché (94.8 MB)
2007: ya en caché (94.1 MB)
2008: ya en caché (93.2 MB)
2009: ya en caché (92.9 MB)
2010: ya en caché (92.9 MB)
2011: ya en caché (93.1 MB)
2012: ya en caché (93.9 MB)
2013: ya en caché (91.1 MB)
2014: ya en caché (95.0 MB)
2015: ya en caché (93.6 MB)
2016: ya en caché (88.5 MB)
2017: ya en caché (96.3 MB)
2018: ya en caché (93.5 MB)
2019: ya en caché (90.6 MB)
2020: ya en caché (91.9 MB)
2021: ya en caché (90.4 MB)
2022: ya en caché (91.1 MB)
2023: ya en caché (91.0 MB)
2024: ya en caché (89.2 MB)
2025: descargando...
2025: listo en 234 s (8761 pasos)

=== Extrayendo 2 metre temperature ===
2000: ya en caché (147.8 MB)
2001: ya en caché (147.4 MB)
2002: ya en caché (147.4 MB)
2003: ya en caché (147.3 MB)
2004: ya en caché (147.6 MB)
2005: ya en caché (1

## 7. Agregación diaria (`ds_diario`)

Crea un NetCDF diario por variable a partir de la caché horaria y los deja en el diccionario `ds_diario`.

| Variable | Estadísticos diarios | Unidad |
|---|---|---|
| `tp` | suma | mm day-1 |
| `t2m` | media, mín, máx | °C |
| `d2m` | media | °C |
| `u10`, `v10` | media | m/s |
| `ws10` (derivada) | media, máx | m/s |
| `hr` (derivada) | media, mín | % |

Notas:
- **Frontera del día**: por defecto días UTC. Con `DESFASE_UTC_HORAS = -4` los días van en hora estándar de Chile continental.
- **Precipitación**: en ERA5 el valor de las HH:00 es la lluvia de la hora anterior, así que el día D suma 01:00 de D a 00:00 de D+1. La caché de `tp` ya incluye las 00:00 del 1-ene siguiente, por lo que con `DESFASE_UTC_HORAS = 0` el 31-dic de `ANIO_FIN` queda completo.
- Con `DESFASE_UTC_HORAS = -4` (en todas las variables) el último día del período queda incompleto y se descarta.
- Solo se guardan días con las 24 horas completas y sin NaN en la grilla.
- `ws10` y `hr` se calculan hora a hora y luego se agregan. HR con Magnus (Alduchov & Eskridge 1996), recortada a 100 %.
- Longitudes de salida en −180..180.

In [11]:
from dask.diagnostics import ProgressBar

assert PASO_HORAS == 1, 'La agregación diaria requiere serie horaria completa'

DIR_DIARIO = DIR_SALIDA / 'diario_era5_chile'
DIR_DIARIO.mkdir(parents=True, exist_ok=True)

# Restos de una ejecución abortada
for f in DIR_DIARIO.glob('*.tmp.nc'):
    f.unlink()

# Frontera del día. ERA5 viene en UTC.
#   0  -> días UTC (00:00-23:00 UTC)
#  -4  -> días en hora estándar de Chile continental (00:00-23:00 UTC-4)
DESFASE_UTC_HORAS = 0

# Estadísticos diarios por variable
ESTADISTICOS = {
    'tp':   ['suma'],                    # mm/día
    't2m':  ['media', 'min', 'max'],     # °C
    'd2m':  ['media'],                   # °C
    'u10':  ['media'],                   # m/s
    'v10':  ['media'],                   # m/s
    'ws10': ['media', 'max'],            # derivada: rapidez del viento a 10 m
    'hr':   ['media', 'min'],            # derivada: humedad relativa a 2 m
}

# Las derivadas se calculan HORA A HORA y recién después se agregan al día
# (la media diaria de la rapidez no es la rapidez de las componentes medias).
VAR_DERIVADAS = {
    'ws10': {'fuentes': ['u10', 'v10'], 'long_name': '10 metre wind speed', 'units': 'm/s'},
    'hr':   {'fuentes': ['t2m', 'd2m'], 'long_name': '2 metre relative humidity', 'units': '%'},
}

VARIABLES_DIARIAS = list(ESTADISTICOS)
print(f'Variables diarias: {VARIABLES_DIARIAS}')
print(f'Salida: {DIR_DIARIO.resolve()}')

Variables diarias: ['tp', 't2m', 'd2m', 'u10', 'v10', 'ws10', 'hr']
Salida: .\diario_era5_chile


In [12]:
def abrir_horario(variable: str) -> xr.DataArray:
    '''Abre en modo lazy todos los años en caché de una variable base (nombre corto: tp, t2m, ...).'''
    cfg = VAR_CFG[variable]
    archivos = [DIR_CACHE / f'{cfg["cache_prefix"]}{a}.nc' for a in range(ANIO_INI, ANIO_FIN + 1)]
    faltan = [f.name for f in archivos if not f.exists()]
    if faltan:
        raise FileNotFoundError(f'Faltan archivos en caché para {variable}: {faltan}')
    ds_h = xr.open_mfdataset(
        archivos,
        combine='by_coords',
        chunks={},  # chunks del disco; se re-agrupan en días completos más abajo
        data_vars='minimal', coords='minimal', compat='override',
    )
    return ds_h[cfg['short_name']]


def es_magnus(T):
    '''Presión de vapor de saturación (hPa), Magnus (Alduchov & Eskridge 1996). T en °C.'''
    return 6.1094 * np.exp(17.625 * T / (T + 243.04))


def serie_horaria(variable: str) -> xr.DataArray:
    '''Serie horaria de una variable base o derivada.'''
    if variable == 'ws10':
        u, v = abrir_horario('u10'), abrir_horario('v10')
        da = np.sqrt(u ** 2 + v ** 2)
    elif variable == 'hr':
        t, td = abrir_horario('t2m'), abrir_horario('d2m')
        da = (100 * es_magnus(td) / es_magnus(t)).clip(max=100)
    else:
        da = abrir_horario(variable)
    return da.rename(variable)


def alinear_dias(da: xr.DataArray, variable: str) -> xr.DataArray:
    '''Desplaza el eje de tiempo a la frontera de día elegida y deja solo días con 24 horas
    y sin NaN en la grilla.

    tp en ERA5 es acumulado de la hora que TERMINA en el timestamp: el valor de las 00:00
    del día D+1 es lluvia del día D. Por eso tp se corre 1 h hacia atrás.
    '''
    desfase = pd.Timedelta(hours=DESFASE_UTC_HORAS)
    if variable == 'tp':
        desfase -= pd.Timedelta(hours=1)
    da = da.assign_coords(time=da.time + desfase)

    tiempos = pd.DatetimeIndex(da.time.values)
    if tiempos.has_duplicates:
        raise ValueError(f'{variable}: hay timestamps duplicados en la caché')
    dias = tiempos.floor('D')
    horas_por_dia = pd.Series(1, index=dias).groupby(level=0).size()

    # días con algún NaN en la grilla (p. ej. horas descargadas antes de que ERA5 las publicara)
    hay_nan = da.isnull().any(dim=['latitude', 'longitude']).compute().values
    dias_nan = pd.DatetimeIndex(dias[hay_nan]).unique()

    en_rango_mask = (horas_por_dia.index.year >= ANIO_INI) & (horas_por_dia.index.year <= ANIO_FIN)
    dias_ok = horas_por_dia.index[
        (horas_por_dia == 24) & en_rango_mask & ~horas_por_dia.index.isin(dias_nan)
    ]
    en_rango = horas_por_dia.index[en_rango_mask]
    descartados = en_rango.difference(dias_ok)
    if len(descartados):
        print(f'  {variable}: {len(descartados)} día(s) incompleto(s) o con NaN descartado(s): '
              f'{[d.strftime("%Y-%m-%d") for d in descartados[:5]]}'
              f'{" ..." if len(descartados) > 5 else ""}')
    return da.isel(time=np.flatnonzero(dias.isin(dias_ok)))


def agregar_diario(variable: str) -> xr.Dataset:
    '''Dataset diario (lazy) con los estadísticos definidos para la variable.'''
    da = alinear_dias(serie_horaria(variable), variable)
    # bloques de 122 días completos x grilla entera
    da = da.chunk({'time': 24 * 122, 'latitude': -1, 'longitude': -1})
    ventanas = da.coarsen(time=24, boundary='exact')
    funciones = {'suma': ventanas.sum, 'media': ventanas.mean,
                 'min': ventanas.min, 'max': ventanas.max}

    if variable in VAR_DERIVADAS:
        long_name, units = VAR_DERIVADAS[variable]['long_name'], VAR_DERIVADAS[variable]['units']
    else:
        long_name, units = VAR_CFG[variable]['long_name'], VAR_CFG[variable]['units']

    ds_d = xr.Dataset()
    for est in ESTADISTICOS[variable]:
        nombre = f'{variable}_{est}'
        ds_d[nombre] = funciones[est]().astype('float32')
        ds_d[nombre].attrs = {
            'long_name': f'Daily {est} of {long_name}',
            'units': f'{units} day-1' if est == 'suma' else units,
            'estadistico': est,
        }

    # tiempo = fecha del día; longitudes a -180..180
    ds_d = ds_d.assign_coords(time=ds_d.time.dt.floor('D'))
    ds_d = ds_d.assign_coords(longitude=((ds_d.longitude + 180) % 360) - 180).sortby('longitude')
    ds_d.attrs = {
        'title': f'ERA5 diario - {variable} - Chile {ANIO_INI}-{ANIO_FIN}',
        'source': 'ERA5 (ARCO, Google Cloud), agregado desde datos horarios',
        'frontera_dia': f'UTC{DESFASE_UTC_HORAS:+d}',
        'nota_tp': 'tp diario = suma de acumulados horarios 01:00..00:00(D+1)' if variable == 'tp' else '',
    }
    return ds_d


def guardar_diario(variable: str, sobrescribir: bool = False) -> Path:
    destino = DIR_DIARIO / f'{variable}_diario_{ANIO_INI}_{ANIO_FIN}.nc'
    if destino.exists() and not sobrescribir:
        print(f'{variable}: ya existe ({destino.stat().st_size / 1e6:.1f} MB)')
        return destino
    print(f'{variable}: agregando a diario...')
    t0 = time.time()
    ds_d = agregar_diario(variable)
    enc = {v: {'zlib': True, 'complevel': 4, 'dtype': 'float32'} for v in ds_d.data_vars}
    tmp = destino.with_suffix('.tmp.nc')
    # compute=False + computar_con_reintentos: barra de progreso del cluster distribuido y reintentos
    delayed = ds_d.to_netcdf(tmp, engine='netcdf4', encoding=enc, compute=False)
    computar_con_reintentos(delayed)
    tmp.replace(destino)
    print(f'{variable}: listo en {time.time() - t0:.0f} s '
          f'({ds_d.sizes["time"]} días, {destino.stat().st_size / 1e6:.1f} MB)')
    return destino


print('Funciones de agregación diaria definidas.')

Funciones de agregación diaria definidas.


In [13]:
for variable in VARIABLES_DIARIAS:
    guardar_diario(variable)

tp: agregando a diario...
tp: listo en 204 s (9497 días, 746.9 MB)
t2m: ya existe (2669.9 MB)
d2m: ya existe (1025.9 MB)
u10: ya existe (1223.3 MB)
v10: ya existe (1165.7 MB)
ws10: ya existe (2308.9 MB)
hr: ya existe (2009.3 MB)


## 8. Lectura y validación de los resultados diarios

In [15]:
# Un Dataset diario por variable, abierto en modo lazy
ds_diario = {
    variable: xr.open_dataset(DIR_DIARIO / f'{variable}_diario_{ANIO_INI}_{ANIO_FIN}.nc', chunks={})
    for variable in VARIABLES_DIARIAS
}

for variable, ds_d in ds_diario.items():
    print(f'{variable:5s} {list(ds_d.data_vars)}  '
          f'{str(ds_d.time.values[0])[:10]} -> {str(ds_d.time.values[-1])[:10]}  ({ds_d.sizes["time"]} días)')

# Comprobaciones de coherencia
t = ds_diario['t2m']
assert bool((t.t2m_min <= t.t2m_media).all()) and bool((t.t2m_media <= t.t2m_max).all()), 't2m: min <= media <= max'
hr = ds_diario['hr'].hr_media
assert float(hr.min()) > 0 and float(hr.max()) <= 100, 'hr_media fuera de (0, 100]'
ws = ds_diario['ws10'].ws10_media
uv = np.hypot(ds_diario['u10'].u10_media, ds_diario['v10'].v10_media)
assert bool((ws >= uv - 1e-4).all()), 'ws10_media < hypot(u10_media, v10_media)'
lon = ds_diario['tp'].longitude
assert float(lon.min()) >= -77 and float(lon.max()) <= -66, 'longitudes fuera de -77..-66'
print('Comprobaciones OK.')

tp    ['tp_suma']  2000-01-01 -> 2025-12-31  (9497 días)
t2m   ['t2m_media', 't2m_min', 't2m_max']  2000-01-01 -> 2025-12-31  (9497 días)
d2m   ['d2m_media']  2000-01-01 -> 2025-12-31  (9497 días)
u10   ['u10_media']  2000-01-01 -> 2025-12-31  (9497 días)
v10   ['v10_media']  2000-01-01 -> 2025-12-31  (9497 días)
ws10  ['ws10_media', 'ws10_max']  2000-01-01 -> 2025-12-31  (9497 días)
hr    ['hr_media', 'hr_min']  2000-01-01 -> 2025-12-31  (9497 días)
Comprobaciones OK.


## 9. Cerrar el cluster

Libera los hilos del scheduler local.

In [16]:
client.close()
print('Cluster cerrado.')

Cluster cerrado.
